In [271]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn import metrics

import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/tolkyn.zhagipar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/tolkyn.zhagipar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/tolkyn.zhagipar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [273]:
df = pd.read_csv('FakeNewsNet.csv')
df.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [275]:
df.shape

(23196, 5)

In [277]:
df = df.drop_duplicates()

In [279]:
df = df.drop_duplicates(subset=['title'], keep='first')

In [281]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(21724, 5)

In [283]:
df.isnull().sum()

title              0
news_url         327
source_domain    327
tweet_num          0
real               0
dtype: int64

In [285]:
df = df.dropna(how='any',axis=0) 
df.shape

(21397, 5)

In [287]:
def preprocess_text(df, column_name):
    """    
    Steps:
    1. Lowercasing
    2. Removing punctuation
    3. Removing numbers
    4. Tokenization
    5. Removing stopwords
    6. Lemmatization
    """
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def clean_text(text):
        text = text.lower()  # Convert to lowercase
        text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation & numbers
        tokens = word_tokenize(text)  # Tokenization
        tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
        tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
        return ' '.join(tokens)

    df['title'] = df['title'].astype(str).apply(clean_text)
    return df

df = preprocess_text(df, df['title'])
df.head()

,title,news_url,source_domain,tweet_num,real
0,kandi burruss explodes rape accusation real ho...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,people choice award best red carpet look,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,sophia bush sends sweet birthday message one t...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,colombian singer maluma spark rumour inappropr...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,gossip girl year later upper east siders shock...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [289]:
y = df['real']
x = df['title']
X = np.array(x)
X

array(['kandi burruss explodes rape accusation real housewife atlanta reunion video',
       'people choice award best red carpet look',
       'sophia bush sends sweet birthday message one tree hill costar hilarie burton breyton eva',
       ...,
       'zayn malik gigi hadids shocking split there chance theyll reunite line',
       'jessica chastain recall moment mother boyfriend slapped kicked genitals',
       'kelly clarkson performs medley kendrick lamars humble hit billboard music award'],
      dtype=object)

In [291]:
x_train, y_train, x_test, y_test = train_test_split(X, y, test_size = 0.2, random_state = 7)
print(x_train.shape, y_train.shape)

(17117,) (4280,)


In [293]:
x_train = [str(x) for x in x_train]
x_test = [str(x) for x in x_test]

#Tf-Idf vectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(X)

tfidf_train, tfidf_test, y_train, y_test = train_test_split(
    tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f'tfidf_train shape:', tfidf_train.shape) 
print(f'tfidf_test shape:', tfidf_test.shape) 
print(f'y_train shape:', y_train.shape) 

tfidf_train shape: (17117, 16775)
tfidf_test shape: (4280, 16775)
y_train shape: (17117,)


In [297]:
naive_bayes_classifier = MultinomialNB()
naive_bayes_classifier.fit(tfidf_train, y_train)

MultinomialNB()

In [299]:
y_pred = naive_bayes_classifier.predict(tfidf_test)
score=metrics.accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')
print (metrics. classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Accuracy: 81.21%
              precision    recall  f1-score   support

    Positive       0.87      0.23      0.36       999
    Negative       0.81      0.99      0.89      3281

    accuracy                           0.81      4280
   macro avg       0.84      0.61      0.63      4280
weighted avg       0.82      0.81      0.77      4280



In [301]:
#CountVectorizer 
count_vectorizer = CountVectorizer()
count_vec = tfidf_vectorizer.fit_transform(X)

In [303]:
count_train, count_test, y_train, y_test = train_test_split(
    count_vec, y, test_size=0.2, random_state=42, stratify=y
)

print(f'count_train shape:', tfidf_train.shape) 
print(f'count_test shape:', tfidf_test.shape) 
print(f'y_train shape:', y_train.shape) 

count_train shape: (17117, 16775)
count_test shape: (4280, 16775)
y_train shape: (17117,)


In [305]:
naive_bayes_classifier = MultinomialNB()
naive_bayes_classifier.fit(count_train, y_train)

MultinomialNB()

In [307]:
y_pred = naive_bayes_classifier.predict(count_test)
score=metrics.accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')
print (metrics. classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Accuracy: 81.21%
              precision    recall  f1-score   support

        Real       0.87      0.23      0.36       999
        Fake       0.81      0.99      0.89      3281

    accuracy                           0.81      4280
   macro avg       0.84      0.61      0.63      4280
weighted avg       0.82      0.81      0.77      4280

